# Non-private promoter BiTFiT

**Curated extraction from the original 124-cell notebook.** Outputs removed. This notebook has **not been executed end-to-end**; original cells are preserved where noted. Training is costly and changes checkpoints. This baseline writes a checkpoint to the original Drive results directory.

## Environment and data
Run in **Google Colab with GPU**. Restart runtime after installing if incompatible libraries are already imported. Check paths before training; original source saves to Google Drive.

In [ ]:
%pip install -q "transformers==4.49.0" "tokenizers==0.21.4" "huggingface-hub==0.36.2" "datasets==3.6.0" "opacus==1.6.0" "pandas==2.2.3" "accelerate" "scikit-learn" "matplotlib" "tqdm" "jedi>=0.16"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/DNABERT_DP_Project/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import random,numpy as np,torch
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
MODEL_ID = 'zhihan1996/DNABERT-2-117M'
class DNABERTBiTFiTClassifier(nn.Module):
    def __init__(self, encoder, num_labels=2):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters(): p.requires_grad = False
        for name,p in self.encoder.named_parameters():
            if name.endswith('.bias'): p.requires_grad = True
        self.classifier = nn.Linear(self.encoder.config.hidden_size,num_labels)
    def forward(self,input_ids,attention_mask=None):
        outputs=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        hidden=outputs[0]
        if attention_mask is not None:
            mask=attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
        else: pooled=hidden.mean(1)
        return self.classifier(pooled)
def fresh_encoder():
    encoder=AutoModel.from_pretrained(MODEL_ID,trust_remote_code=True,add_pooling_layer=False)
    for module in encoder.modules():
        if module.__class__.__name__=='BertUnpadSelfAttention':
            module.p_dropout=0.1
            if hasattr(module,'dropout'): module.dropout.p=0.1
    return encoder

**Original source cell 68** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 50 - RESTORE ORIGINAL PROMOTER DATASET
# Recreate the original 42,620 / 4,736 train/validation split
# ============================================================

import os
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

MODEL_ID = "zhihan1996/DNABERT-2-117M"
SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 8

print("=" * 68)
print("CELL 50 - RESTORE PROMOTER DATASET")
print("=" * 68)

# ------------------------------------------------------------
# 1. Load original GUE promoter dataset
# ------------------------------------------------------------

print("\n1. LOADING GUE PROMOTER DATASET")
print("-" * 68)

promoter = load_dataset(
    "leannmlindsey/GUE",
    "prom_core_all"
)

print(promoter)

original_train = promoter["train"]
original_test = promoter["test"]

print("\nOriginal training examples:", len(original_train))
print("Original test examples:", len(original_test))

assert len(original_train) == 47356
assert len(original_test) == 5920

# ------------------------------------------------------------
# 2. Recreate stratified train/validation split
# ------------------------------------------------------------

print("\n2. RECREATING TRAIN/VALIDATION SPLIT")
print("-" * 68)

indices = np.arange(len(original_train))
labels = np.asarray(original_train["label"])

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

promoter_train = original_train.select(train_idx.tolist())
promoter_val = original_train.select(val_idx.tolist())
promoter_test = original_test

print("Training:", len(promoter_train))
print("Validation:", len(promoter_val))
print("Test:", len(promoter_test))

assert len(promoter_train) == 42620
assert len(promoter_val) == 4736
assert len(promoter_test) == 5920

# ------------------------------------------------------------
# 3. Load tokenizer
# ------------------------------------------------------------

print("\n3. TOKENIZER")
print("-" * 68)

promoter_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True
)

print("Tokenizer:", promoter_tokenizer.__class__.__name__)

# ------------------------------------------------------------
# 4. Tokenize all three splits
# ------------------------------------------------------------

def tokenize_promoter(batch):

    encoded = promoter_tokenizer(
        batch["sequence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    encoded["labels"] = batch["label"]

    return encoded


promoter_train_dataset = promoter_train.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_train.column_names,
    desc="Tokenizing promoter training"
)

promoter_val_dataset = promoter_val.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_val.column_names,
    desc="Tokenizing promoter validation"
)

promoter_test_dataset = promoter_test.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_test.column_names,
    desc="Tokenizing promoter test"
)

# ------------------------------------------------------------
# 5. Build evaluation loaders
# ------------------------------------------------------------

promoter_collator = DataCollatorWithPadding(
    tokenizer=promoter_tokenizer,
    padding=True,
    return_tensors="pt"
)

# IMPORTANT:
# This ordinary DataLoader is for inspection/evaluation.
# It is NOT the Poisson sampler for private training.

promoter_val_loader = DataLoader(
    promoter_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

promoter_test_loader = DataLoader(
    promoter_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n4. RESTORED DATASET VERIFICATION")
print("-" * 68)

print("Training examples:", len(promoter_train_dataset))
print("Validation examples:", len(promoter_val_dataset))
print("Test examples:", len(promoter_test_dataset))

print(
    "Training label distribution:",
    np.bincount(
        np.asarray(promoter_train_dataset["labels"])
    ).tolist()
)

print(
    "Validation label distribution:",
    np.bincount(
        np.asarray(promoter_val_dataset["labels"])
    ).tolist()
)

sample = promoter_train_dataset[0]

print("Tokenized fields:", list(sample.keys()))
print("First sequence token count:", len(sample["input_ids"]))

assert len(promoter_train_dataset) == 42620
assert len(promoter_val_dataset) == 4736
assert len(promoter_test_dataset) == 5920

print("\n" + "=" * 68)
print("CELL 50 COMPLETE - PROMOTER DATASET RESTORED")
print("=" * 68)

In [ ]:
# Compatibility aliases used by the original DP-SGD training cells.
train_dataset = promoter_train_dataset
val_dataset = promoter_val_dataset
test_dataset = promoter_test_dataset
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,collate_fn=promoter_collator)
eval_loader = promoter_val_loader
test_loader = promoter_test_loader
classifier_model = DNABERTBiTFiTClassifier(fresh_encoder(),num_labels=2).to(device)
print('Training:',len(train_dataset),'Validation:',len(val_dataset),'Test:',len(test_dataset))

**Static audit correction:** The compatibility DataLoader now uses `promoter_collator` defined by the dataset setup cell, replacing an undefined `collator` name. This notebook has not been run end-to-end.


## Original experiment cells
**CAUTION:** The following cells were extracted from different Colab sessions. Review paths, variable dependencies, privacy assumptions, and checkpoint names before running. Run one training budget at a time.

**Original source cell 46** — retained with its original logic.


In [ ]:
# Cell 34 - Train and Save Non-Private BiTFiT Baseline

import os
import gc
import json
import time
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoModel
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

MODEL_ID = "zhihan1996/DNABERT-2-117M"
RESULTS_DIR = "/content/drive/MyDrive/DNABERT_DP_Project/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
EPOCHS = 3
LEARNING_RATE = 2e-4

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

# --------------------------------------------------
# 2. Fresh pretrained encoder
# --------------------------------------------------

encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

# Apply DNABERT-2 attention compatibility patch
for module in encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        if hasattr(module, "dropout"):
            module.dropout.p = 0.1

# Build original BiTFiT classifier
baseline_model = DNABERTBiTFiTClassifier(
    encoder=encoder,
    num_labels=2
).to(device)

# --------------------------------------------------
# 3. Optimizer and loss
# --------------------------------------------------

trainable_parameters = [
    p for p in baseline_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE
)

criterion = nn.CrossEntropyLoss()

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_parameters)
)

# --------------------------------------------------
# 4. Training
# --------------------------------------------------

history = []
best_accuracy = -1.0
best_epoch = None

best_checkpoint_path = os.path.join(
    RESULTS_DIR,
    "dnabert_nonprivate_best.pt"
)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    baseline_model.train()

    total_train_loss = 0.0
    total_train_examples = 0

    for batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} Training"
    ):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = (
            outputs.logits
            if hasattr(outputs, "logits")
            else outputs
        )

        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_train_loss += loss.item() * batch_size
        total_train_examples += batch_size

    avg_train_loss = (
        total_train_loss / total_train_examples
    )

    # ----------------------------------------------
    # Validation
    # ----------------------------------------------

    baseline_model.eval()

    val_losses = []
    val_predictions = []
    val_labels = []

    with torch.inference_mode():

        for batch in tqdm(
            eval_loader,
            desc=f"Epoch {epoch}/{EPOCHS} Validation"
        ):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = baseline_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = (
                outputs.logits
                if hasattr(outputs, "logits")
                else outputs
            )

            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            loss = criterion(logits, labels)

            val_losses.append(
                (loss.item(), labels.size(0))
            )

            predictions = torch.argmax(
                logits, dim=-1
            )

            val_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

            val_labels.extend(
                labels.cpu().numpy().tolist()
            )

    avg_val_loss = (
        sum(loss * n for loss, n in val_losses)
        / sum(n for _, n in val_losses)
    )

    val_accuracy = accuracy_score(
        val_labels, val_predictions
    )

    elapsed = (time.time() - start_time) / 60

    print(
        f"\nEpoch {epoch}/{EPOCHS}"
        f" | Train Loss: {avg_train_loss:.4f}"
        f" | Val Loss: {avg_val_loss:.4f}"
        f" | Val Accuracy: {val_accuracy:.4f}"
        f" | Time: {elapsed:.1f} min"
    )

    history.append({
        "epoch": epoch,
        "train_loss": float(avg_train_loss),
        "val_loss": float(avg_val_loss),
        "val_accuracy": float(val_accuracy),
        "elapsed_minutes": float(elapsed)
    })

    # Save best checkpoint
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_epoch = epoch

        state_dict = {
            key: value.detach().cpu().clone()
            for key, value in baseline_model.state_dict().items()
        }

        torch.save(
            state_dict,
            best_checkpoint_path
        )

        del state_dict

        print("Best checkpoint saved to Google Drive.")

# --------------------------------------------------
# 5. Save final model and results
# --------------------------------------------------

final_checkpoint_path = os.path.join(
    RESULTS_DIR,
    "dnabert_nonprivate_final.pt"
)

torch.save(
    {
        key: value.detach().cpu().clone()
        for key, value in baseline_model.state_dict().items()
    },
    final_checkpoint_path
)

results = {
    "method": "Non-private BiTFiT",
    "model": MODEL_ID,
    "dataset": "GUE prom_core_all",
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "best_epoch": best_epoch,
    "best_validation_accuracy": float(best_accuracy),
    "total_runtime_minutes": (
        time.time() - start_time
    ) / 60
}

with open(
    os.path.join(
        RESULTS_DIR,
        "results_nonprivate.json"
    ),
    "w"
) as f:
    json.dump(results, f, indent=2)

pd.DataFrame(history).to_csv(
    os.path.join(
        RESULTS_DIR,
        "history_nonprivate.csv"
    ),
    index=False
)

print("\nNON-PRIVATE BASELINE COMPLETE")
print("-" * 60)
print(f"Best epoch: {best_epoch}")
print(f"Best validation accuracy: {best_accuracy:.4%}")
print(f"Total runtime: {results['total_runtime_minutes']:.1f} min")
print(f"Checkpoints saved to: {RESULTS_DIR}")

# Release GPU memory
del baseline_model, encoder, optimizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Verification status
- Notebook JSON and code-cell Python syntax were checked (excluding Colab magics).
- **No GPU training or end-to-end runtime verification was performed.**
- Checkpoint references and privacy accounting are retained from original experiments, not independently certified.
- Avoid rerunning if a completed checkpoint exists.